<a href="https://colab.research.google.com/github/vyasaastik/Another-backend4/blob/main/Story_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tiktoken

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.5 MB/s eta 0:00:00


In [2]:
## Load and Clean the Dataset

# Load dataset
with open(r'/content/Rag-dataset.txt', 'r', encoding='utf-8') as f:
    final_data = f.read()

# Clean dataset
final_data = final_data.replace('\n', ' ').replace('\r', ' ').strip()


In [3]:
## Tokenization

# Load GPT-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Encode text
tokens = tokenizer.encode(final_data)
data = torch.tensor(tokens, dtype=torch.long)

print(f"Total tokens: {len(data)}")


Total tokens: 11818


In [4]:
## Dataset and DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, block_size=96, stride=4):
        self.inputs, self.targets = [], []
        for i in range(0, len(data) - block_size - 1, stride):
            x = data[i:i+block_size]
            y = data[i+1:i+block_size+1]
            self.inputs.append(x)
            self.targets.append(y)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Create dataset and dataloader
block_size = 96
stride = 4
batch_size = 4

dataset = GPTDataset(data, block_size=block_size, stride=stride)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [5]:
## Input Embeddings

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)

    def forward(self, x):
        if x.ndim == 3:
            x = x.squeeze(1)
        B, T = x.shape
        token_embeds = self.token_embedding(x)
        position_ids = torch.arange(T, device=x.device).unsqueeze(0)
        position_embeds = self.position_embedding(position_ids)
        return token_embeds + position_embeds


In [6]:
## Multi-Head Attention

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert embedding_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_size = embedding_dim // num_heads

        self.query = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key   = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.value = nn.Linear(embedding_dim, embedding_dim, bias=False)

        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        k = self.key(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = self.value(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_size ** 0.5)
        att = att.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        return out


In [7]:
## Feed-Forward

class FeedForward(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
## Transformer Block

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attn = MultiHeadAttention(embedding_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.ffwd = FeedForward(embedding_dim, embedding_dim * 4, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [9]:
## GPT-Model

class GPTModel(nn.Module):
    def __init__(self, vocab_size, block_size, embedding_dim=192, num_heads=6, num_layers=8, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.embedding = InputEmbedding(vocab_size, embedding_dim, block_size)

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(embedding_dim, num_heads, block_size, dropout) for _ in range(num_layers)]
        )

        self.ln_f = nn.LayerNorm(embedding_dim)
        self.output_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        x = self.embedding(idx)
        x = self.transformer_blocks(x)
        x = self.ln_f(x)
        logits = self.output_head(x)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)
        return logits, loss


In [10]:
## Create Model and Optimizer

vocab_size = tokenizer.n_vocab

model = GPTModel(vocab_size, block_size, embedding_dim=192, num_heads=6, num_layers=8, dropout=0.1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)


In [11]:
## Train the Model

epochs = 10

for epoch in range(1, epochs + 1):
    total_loss = 0
    model.train()

    for batch_idx, (x, y) in enumerate(dataloader):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"✅ Epoch {epoch} completed. Avg Loss: {avg_loss:.4f}")


✅ Epoch 1 completed. Avg Loss: 3.3966
✅ Epoch 2 completed. Avg Loss: 0.4170
✅ Epoch 3 completed. Avg Loss: 0.1574
✅ Epoch 4 completed. Avg Loss: 0.1262
✅ Epoch 5 completed. Avg Loss: 0.1097
✅ Epoch 6 completed. Avg Loss: 0.0994
✅ Epoch 7 completed. Avg Loss: 0.0945
✅ Epoch 8 completed. Avg Loss: 0.0875
✅ Epoch 9 completed. Avg Loss: 0.0826
✅ Epoch 10 completed. Avg Loss: 0.0765


In [12]:
## Generate Using Top-k

@torch.no_grad()
def generate_top_k(model, start_text, tokenizer, max_new_tokens=100, k=20):
    model.eval()
    device = next(model.parameters()).device

    input_ids = tokenizer.encode(start_text)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        input_crop = input_ids[:, -model.block_size:]
        logits, _ = model(input_crop)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)

        probs, indices = torch.topk(probs, k, dim=-1)
        probs = probs / probs.sum(dim=-1, keepdim=True)

        next_token = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
        next_token = indices.gather(-1, next_token)           # (batch_size, 1)

        # ⭐ No squeeze needed anymore!

        input_ids = torch.cat((input_ids, next_token), dim=1)

    return tokenizer.decode(input_ids[0].tolist())




In [13]:
## Example
prompt = "The boy had always believed the forest was alive, but tonight, he knew it for certain."
generated_text = generate_top_k(model, prompt, tokenizer, max_new_tokens=200, k=20)
print("Generated Text:\n", generated_text)


Generated Text:
 The boy had always believed the forest was alive, but tonight, he knew it for certain. He made a very self-possessed young lady of fifteen; "in the meantime you must try and put up with me."  Framton Nuttel endeavored to say the correct something which should duly flatter the niece of the moment without unduly discounting the aunt that was to come. Privately he doubted more than ever whether these formal visits on a succession of total strangers would do.  "It is that window wide open window if day, when I should duly flitted. It was square beside Mrs. Gisburn--had not dis years before, to have disarming, it became the to study one calm without wonderty touch.  The night air of Jack's "strongest corner of being crowned back his life which she suddenly without an exqu of Jack--since he had been surrounded.  "Her great rudeness began to " letters of my most egregious thing--rather moved aside?" I said briefly stood there.  "Not for a straw beating tumultuously


## Finetune the Model on Multi-Genre Dataset

In [14]:
## Save to file
with open(r"/content/genre-dataset.txt" , 'r' , encoding='utf-8') as f:
  raw_data = f.read()

In [15]:
## Clean the dataset
cleaned_data = raw_data.replace('\n', ' ').replace('\r', ' ').strip()


# Tokenization
import tiktoken

# Load GPT-2 Tokenizer
tokenizer = tiktoken.get_encoding('gpt2')

## Tokenize the multi-genre data
tokens = tokenizer.encode(cleaned_data)
data = torch.tensor(tokens , dtype=torch.long)

print(f'Total tokens: {len(data)}')

Total tokens: 15156


In [16]:
## Create Dataset and Dataloader
class GPTDataset(Dataset):
    def __init__(self, data, block_size=96, stride=4):
        self.inputs, self.targets = [], []
        for i in range(0, len(data) - block_size - 1, stride):
            x = data[i:i+block_size]
            y = data[i+1:i+block_size+1]
            self.inputs.append(x)
            self.targets.append(y)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Create the dataset and dataloader
block_size = 96
stride = 4
batch_size = 4

dataset = GPTDataset(data , block_size=block_size, stride=stride)
dataloader = DataLoader(dataset , batch_size=batch_size , shuffle=True)


In [17]:
## Build the model
vocab_size = tokenizer.n_vocab
block_size = 96

# Create a slightly bigger GPT Model
model = GPTModel(
    vocab_size = vocab_size,
    block_size=block_size,
    embedding_dim = 256,
    num_heads = 8,
    num_layers = 10,
    dropout = 0.1
)

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# optimizer
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4)



In [18]:
## Finetune the model

# Fine-tune for a few epochs
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    model.train()

    for batch_idx, (x, y) in enumerate(dataloader):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    avg_loss = total_loss / len(dataloader)
    print(f"Fine-tune Epoch {epoch+1} completed. Avg Loss: {avg_loss:.4f}")


Fine-tune Epoch 1 completed. Avg Loss: 6.1942
Fine-tune Epoch 2 completed. Avg Loss: 3.6169
Fine-tune Epoch 3 completed. Avg Loss: 2.0849
Fine-tune Epoch 4 completed. Avg Loss: 1.0859
Fine-tune Epoch 5 completed. Avg Loss: 0.4475
Fine-tune Epoch 6 completed. Avg Loss: 0.2097
Fine-tune Epoch 7 completed. Avg Loss: 0.1332
Fine-tune Epoch 8 completed. Avg Loss: 0.0981
Fine-tune Epoch 9 completed. Avg Loss: 0.0802
Fine-tune Epoch 10 completed. Avg Loss: 0.0684


# Save the model and optimizer

In [20]:
import os

# Create the directory if it doesn't exist
os.makedirs('saved_model', exist_ok=True)

# Save model after fine-tuning
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}, 'saved_model/improved_finetuned_model.pth')

print("Model and optimizer saved successfully!")

Model and optimizer saved successfully!


In [28]:
### Load your model

import torch
import gradio as gr

# Load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_size = tokenizer.n_vocab
block_size = 96

model = GPTModel(vocab_size, block_size, embedding_dim=256, num_heads=8, num_layers=10, dropout=0.1)
# Updated the path to match the saved file name
checkpoint = torch.load('saved_model/improved_finetuned_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [23]:
## Gradio
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.5 MB/s eta 0:00:00


In [31]:
### Load your model

import torch
import gradio as gr

# Load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_size = tokenizer.n_vocab
block_size = 96

model = GPTModel(vocab_size, block_size, embedding_dim=256, num_heads=8, num_layers=10, dropout=0.1)
checkpoint = torch.load('saved_model/improved_finetuned_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [ ]:
# Gradio generation function
!pip install --upgrade gradio
import torch
import time
import re

def clean_output(text):
    return re.sub(r"<genre=.*?>", "", text)

# Typing animation with spinner and cleaned output
def generate_story_with_typing(genre, prompt, max_tokens):
    spinner = "⏳ Crafting your story, please wait...\n"
    yield spinner

    full_prompt = f"<genre={genre}> {prompt}"
    generated_text = generate_top_k(model, full_prompt, tokenizer, max_new_tokens=max_tokens, k=20)

    cleaned_text = clean_output(generated_text)

    output_text = ""
    for char in cleaned_text:
        output_text += char
        time.sleep(0.012)  # Typing speed
        yield output_text

# Gradio Interface
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    with gr.Column(elem_id="background"):
        gr.Markdown(
            """
            <div style="text-align: center; animation: fadeIn 2s;">
                <h1 style="font-size: 3.5em; font-weight: bold; color: #00C9FF;">🔮 DreamWeaver AI</h1>
                <p style="font-size: 1.3em; color: #a0a0a0;">Create magical stories instantly with fine-tuned AI ✨</p>
            </div>
            """,
            elem_id="header"
        )

        with gr.Row():
            genre = gr.Dropdown(["horror", "scifi", "classic", "fantasy", "mystery"], label="🎭 Choose Genre")
            tokens = gr.Slider(50, 500, value=200, step=10, label="✍️ Number of Words")

        prompt = gr.Textbox(lines=4, placeholder="Enter your creative idea here...", label="📝 Story Prompt")

        generate_button = gr.Button("🚀 Generate My Story", elem_id="generate-btn")

        output = gr.Textbox(lines=20, label="📖 Your Story", interactive=False)

        generate_button.click(
    fn=generate_story_with_typing,
    inputs=[genre, prompt, tokens],
    outputs=output,
    api_name="generate_story",
    # Remove stream=True
)

    # Custom CSS to add background and styling
    app.css = """ # Fix: Assign CSS directly to app.css attribute
    /* Background and Layout */
    #background {
        background: linear-gradient(to right, #0f2027, #203a43, #2c5364);
        padding: 2rem;
        min-height: 100vh;
    }
    /* Logo Section */
    #header h1 {
        font-family: 'Poppins', sans-serif;
    }
    #header p {
        font-family: 'Open Sans', sans-serif;
    }
    /* Buttons */
    #generate-btn {
        font-size: 1.3em !important;
        padding: 14px 28px !important;
        border-radius: 14px !important;
        background: linear-gradient(to right, #00C9FF, #92FE9D) !important;
        color: black !important;
        font-weight: bold;
        transition: all 0.3s ease-in-out;
        margin-top: 1rem;
    }
    #generate-btn:hover {
        transform: scale(1.05);
        background: linear-gradient(to right, #92FE9D, #00C9FF) !important;
    }
    /* Output Box */
    textarea {
        background: rgba(255,255,255,0.1) !important;
        border: 2px solid #00C9FF !important;
        color: #ffffff !important;
        font-size: 1.1em !important;
        border-radius: 12px !important;
    }
    """

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2cd442ade2653b2d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
